# 一个可以逐个吐token的Model

In [1]:
import torch

# 上一节里我们实现了这样的一个Attention类
class Attention(torch.nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.d_k = torch.tensor(dims)
        
        self.q_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.k_proj = torch.nn.Linear(self.d_k, self.d_k)
        self.v_proj = torch.nn.Linear(self.d_k, self.d_k)

        self.k_cache = None
        self.v_cache = None
        
        self.feed_forward = torch.nn.Sequential(
            torch.nn.Linear(dims, 2*dims),
            torch.nn.ReLU(),
            torch.nn.Linear(2*dims, dims)
        )

        self.norm = torch.nn.LayerNorm(dims)

    def forward(self, x):
        res = x
        Q = self.q_proj(x)
        if self.k_cache is None and self.v_cache is None:
            self.k_cache = self.k_proj(x)
            self.v_cache = self.v_proj(x)
        else:
            self.k_cache = torch.cat([self.k_cache, self.k_proj(x)], dim=1)
            self.v_cache = torch.cat([self.v_cache, self.v_proj(x)], dim=1)

        K = self.k_cache
        V = self.v_cache

        attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)
        attention_weights = torch.softmax(attention_weights/torch.sqrt(self.d_k), dim=-1)
        output = torch.einsum("bqL, bLd->bqd", attention_weights, V)
        output = self.norm(output+res)

        res = output
        output = self.feed_forward(output)
        output = self.norm(output+res)
        
        return output

    def empty_kv_cache(self):
        self.k_cache = None
        self.v_cache = None

In [2]:
# 我们在此基础上要写一个MyModel类，这个MyModel类要能实现自回归输出，那么我们首先定义MyModel框架
# 基本的方法应该有__init__和generate，generate用于自回归输出
class MyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def generate(self, x):
        output = x
        return output

## 思考写这个Model类时需要注意哪些问题

1、怎么从token的id到转化到feature?

2、生成过程是什么样的，怎么从feature再到新的id？

## 思考问题1

1、怎么从token的id到转化到feature?

现在通常的做法是我们有一个预先设定好的词表，这个词表的大小我们设为vocabulary_size，词表每个位置代表的就是一个shape为(dims, )的响亮

那么一个token的id就是这个token在这个词表的位置，用id作为key去词表取出来的值就是这个token的feature。

In [3]:
# 对于问题1的实现

# 我们尝试构建一个词表，我幸运数字是9所以词表大小设成9（叉腰
class MyModel(torch.nn.Module):
    def __init__(self, vocabulary_size=9, dims=16):
        super().__init__()
        self.vocabulary_size=vocabulary_size
        self.dims = dims
        
        self.vocabulary = torch.nn.Embedding(self.vocabulary_size, self.dims)

        # 搞个两层的，之所以用ModuleList不用ModuleSequantial是因为大大概率会在中间做一些处理
        self.layers = torch.nn.ModuleList(
            [Attention(dims=self.dims) for _ in range(2)]
        )

        #分类器就简单了，就是一个Linear+softmax就好了，btw，在***ForCasualLM里，分类器一般都叫做LM_Head
        self.lm_head = torch.nn.Linear(self.dims, self.vocabulary_size, bias=False)
    
    def empty_all_kv_cache(self):
        for layer in self.layers:
            layer.empty_kv_cache()

    def forward(self, input_ids):
        input_features = self.vocabulary(input_ids)

        hidden_states = input_features
        for layer in self.layers:
            hidden_states = layer(hidden_states)

        logits = self.lm_head(hidden_states)
        
        return logits, hidden_states
    
    def generate(self, input_ids):
        pass

In [4]:
#测试一下
model=MyModel()

# 模拟输入，vocabulary的取值最小是0，最大是8
input_ids = torch.randint(0, 9, (2,3))

print(input_ids, input_ids.shape)
print(model.vocabulary(input_ids), model.vocabulary(input_ids).shape)
#可以看得出经过vocabulary映射以后，在原有input_ids的shape后面多了一个dims的维度，就说明把每一个token id（标量）都映射到了一个特征上

tensor([[3, 3, 4],
        [4, 3, 3]]) torch.Size([2, 3])
tensor([[[-0.2602, -2.8784, -1.9239,  1.6872, -0.1760,  1.3079,  0.6061,
          -1.2246, -0.9270,  0.9786,  0.2276, -1.1607, -0.7618, -0.0088,
          -1.1692,  2.4459],
         [-0.2602, -2.8784, -1.9239,  1.6872, -0.1760,  1.3079,  0.6061,
          -1.2246, -0.9270,  0.9786,  0.2276, -1.1607, -0.7618, -0.0088,
          -1.1692,  2.4459],
         [ 1.0929, -1.5646, -0.0233,  1.3431, -0.8911,  0.5626, -0.1648,
           1.3392, -0.8172, -1.1390, -0.2633,  0.1282,  1.6438, -0.0658,
           0.8575,  0.7497]],

        [[ 1.0929, -1.5646, -0.0233,  1.3431, -0.8911,  0.5626, -0.1648,
           1.3392, -0.8172, -1.1390, -0.2633,  0.1282,  1.6438, -0.0658,
           0.8575,  0.7497],
         [-0.2602, -2.8784, -1.9239,  1.6872, -0.1760,  1.3079,  0.6061,
          -1.2246, -0.9270,  0.9786,  0.2276, -1.1607, -0.7618, -0.0088,
          -1.1692,  2.4459],
         [-0.2602, -2.8784, -1.9239,  1.6872, -0.1760,  1.3079,  

In [5]:
print(f"logtis长什么样：\n{model(input_ids)[0]}")
print(f"logits的shape什么样：\n{model(input_ids)[0].shape}")
# 看得出来吼，前一个logits的每一项都是和为1的一个tensor，同时这个shape和输入shape一样，表示以这个位置之前所有token作为历史信息来预测的下一个token的词典概率。
# 那预测的时候需要这么多logits吗？不是的，只有最后一个logits是需要的，因为只有最后一个logtis才是预测的未知未知的token。

logtis长什么样：
tensor([[[ 0.5782, -0.8280, -0.2105,  0.3199,  1.3612,  1.0485, -1.0563,
          -0.4447, -0.5876],
         [ 0.5782, -0.8280, -0.2105,  0.3199,  1.3612,  1.0485, -1.0563,
          -0.4447, -0.5876],
         [ 0.2110, -0.6353, -0.6033,  0.4433,  0.3869,  0.7689, -0.1266,
          -0.9977,  0.1649]],

        [[ 0.2110, -0.6353, -0.6033,  0.4433,  0.3869,  0.7689, -0.1266,
          -0.9977,  0.1649],
         [ 0.5782, -0.8280, -0.2105,  0.3199,  1.3612,  1.0485, -1.0563,
          -0.4447, -0.5876],
         [ 0.5782, -0.8280, -0.2105,  0.3199,  1.3612,  1.0485, -1.0563,
          -0.4447, -0.5876]]], grad_fn=<UnsafeViewBackward0>)
logits的shape什么样：
torch.Size([2, 3, 9])


## 思考问题2

2、生成过程是什么样的，怎么从feature再到新的id？

首先这个过程肯定是会停止的，可能是达到了停止条件，也可能是生成长度达到了最大，如果是后者的话，那就该传入一个max_length参数。

其次我们一次前向传播得到的东西是一个(batch_size, sequence_length, vocabulary_size)的tensor，还要让这个玩意儿转化成index的形式。

我们可以直接把这个过程当作一个分类问题，就是把最终的feature扔进一个vocabulary_size的分类器中，取最大的那个index就是我们预测的下一个token。

In [6]:
# 那么尝试构造一个类看看
class Try_MyModel(MyModel):
    def generate(self, input_ids, max_length):
        new_token=0
        # 在generate前要清空kv cache。
        self.empty_all_kv_cache()
        
        while new_token<=max_length:
            # 这里老token输入进来，然后产生一个新logtis
            # 一些代码
            # 然后logtis里找到最大的，把index取出来当作下一个token
            # 一些代码
            new_token+=1
        # 最后我们应该要返回一个(batch_size, max_length)的tensor
        return None

In [7]:
# 老token输入进来，然后产生一个新logtis的逻辑
# fake一个input
input_ids = torch.randint(0, 9, (2, 1))
print(f'input_ids的shape应该是：{input_ids.shape}')
logits = model(input_ids)[0]
print(f"logits的shape是：{logits.shape}")

input_ids的shape应该是：torch.Size([2, 1])
logits的shape是：torch.Size([2, 1, 9])


In [8]:
# 取最大index
new_input_ids = logits.argmax(dim=-1)
print(new_input_ids, new_input_ids.shape)

tensor([[8],
        [5]]) torch.Size([2, 1])


In [9]:
input_ids = torch.randint(0, 9, (2, 1))
logits = model(input_ids)[0]
new_input_ids = logits.argmax(dim=-1)[:, -1].unsqueeze(1)
new_input_ids, new_input_ids.shape

(tensor([[4],
         [4]]),
 torch.Size([2, 1]))

In [10]:
# 尝试搞进去Try_MyModel里去
class Try_MyModel(MyModel):
    def generate(self, input_ids, max_length=10):
        new_token=0
        self.empty_all_kv_cache()
        
        output_ids = [input_ids,]

        while new_token<max_length:
            logits = self.forward(input_ids=input_ids)[0]

            probs = torch.softmax(logits, dim=-1)
            # 这里的input_ids的shape是(batch_size, length)，在首次输入的时候length不一定为1
            input_ids = probs.argmax(dim=-1) 
            # 提取最后一列，同时保持shape是(batch_size, 1)，替换原有的input_ids
            input_ids = input_ids[:, -1].reshape(-1, 1)
            # 生成一个就append一个，这里不用cat是因为cat会慢，最后一起cat
            
            output_ids.append(input_ids)
            new_token+=1

        output_ids = torch.cat(output_ids, dim=-1)
        return output_ids

In [11]:
model = Try_MyModel()
# fake一个input_ids, 这个input_ids初始只有1个token
input_ids = torch.randint(0, 9, (2, 1))
output_ids = model.generate(input_ids)

print(f"按理说output应该比input多10个token")
print(f"input_ids的shape：{input_ids.shape}")
print(f"output_ids的shape：{output_ids.shape}")

按理说output应该比input多10个token
input_ids的shape：torch.Size([2, 1])
output_ids的shape：torch.Size([2, 11])


In [12]:
# 但是很明显我们输入的时候大概率不是只输入一个token，多token也试一下
input_ids = torch.randint(0, 9, (2, 3))
output_ids = model.generate(input_ids)

# 按理说output也应该比input多10个token
print(f"input_ids的shape：{input_ids.shape}")
print(f"output_ids的shape：{output_ids.shape}")

input_ids的shape：torch.Size([2, 3])
output_ids的shape：torch.Size([2, 13])


In [13]:
# 汇总一下
class MyModel(torch.nn.Module):
    def __init__(self, vocabulary_size=9, dims=16):
        super().__init__()
        self.vocabulary_size=vocabulary_size
        self.dims = dims
        
        self.vocabulary = torch.nn.Embedding(self.vocabulary_size, self.dims)

        # 搞个两层的，之所以用ModuleList不用ModuleSequantial是因为大大概率会在中间做一些处理
        self.layers = torch.nn.ModuleList(
            [Attention(dims=self.dims) for _ in range(2)]
        )

        #分类器就简单了，就是一个Linear+softmax就好了，btw，在***ForCasualLM里，分类器一般都叫做LM_Head
        self.lm_head = torch.nn.Linear(self.dims, self.vocabulary_size, bias=False)
    
    def empty_all_kv_cache(self):
        for layer in self.layers:
            layer.empty_kv_cache()

    def forward(self, input_ids):
        input_features = self.vocabulary(input_ids)

        hidden_states = input_features
        for layer in self.layers:
            hidden_states = layer(hidden_states)

        logits = torch.softmax(self.lm_head(hidden_states), dim=-1)

        return logits, hidden_states
    
    def generate(self, input_ids, max_length=10):
        new_token=0
        self.empty_all_kv_cache()
        
        output_ids = [input_ids,]

        while new_token<max_length:
            logits = self.forward(input_ids=input_ids)[0]

            probs = torch.softmax(logits, dim=-1)
            input_ids = probs.argmax(dim=-1)
            input_ids = input_ids[:, -1].reshape(-1, 1)
            
            output_ids.append(input_ids)
            new_token+=1

        output_ids = torch.cat(output_ids, dim=-1)
        return output_ids

## 我们现在已经搭建起来了一个Model类，思考一下这个Model类还可以在哪里改进？

1、现在我们就直接拿最大值的index当作下一个token的index，那有没有一种可能就是，这个只是局部最优不是全局最优呢？或者说我就单纯看这个不爽，我不想每次都取最大值。

2、我们现在支持的输入token都是一样长的，但是实际情况里句子肯定有长有短的，这要怎么办呢？

我将在[第四节](./04.ipynb)解决新问题1

在[第五节](./05.ipynb)解决新问题2